# M3GNet Kaggle pipeline (two molecules + combined)

This notebook installs dependencies, prepares MD17 XYZ data, trains two single-molecule models plus a combined model, merges checkpoints, and runs switched-embedding evaluations.

In [ ]:
!nvidia-smi

In [ ]:
import os
import subprocess
import sys
import torch

print("torch:", torch.__version__, "cuda:", torch.version.cuda)
cuda_version = torch.version.cuda or ""
if cuda_version.startswith("12."):
    dgl_repo = "https://data.dgl.ai/wheels/cu121/repo.html"
    print("CUDA", cuda_version, "-> using DGL cu121 wheel")
elif cuda_version.startswith("11.8"):
    dgl_repo = "https://data.dgl.ai/wheels/cu118/repo.html"
    print("CUDA", cuda_version, "-> using DGL cu118 wheel")
else:
    dgl_repo = None
    print("CUDA not detected or unsupported; installing CPU DGL.")

if dgl_repo:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "dgl", "-f", dgl_repo])
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "dgl"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matgl", "torchdata==0.7.1"])

os.environ["DGL_SKIP_GRAPHBOLT"] = "1"


In [ ]:
!cp -r /kaggle/input/proj2917/* .

In [ ]:
from pathlib import Path
import os
root = Path("/kaggle/working").resolve()
os.chdir(root)
print("Working directory:", root)
# Toggle HDF5-based ANI/MD datasets
USE_ANI_H5 = False
H5_FILE = Path("M3GNet/data/raw/ani_md_bench.h5")
H5_MOLECULES = ["molCaffeine", "molAcetaminophen"]
# Root folder for raw XYZ/EXTXYZ datasets (MD17/MD22)
RAW_DIR = Path("M3GNet/data/raw")
# Default datasets for multi-merge (MD17-style)
DATASET_SOURCES = [
    "beta-Li3PS4",
    "gamma-Li3PS4",
]
def _sanitize(name: str) -> str:
    out = []
    for ch in name:
        if ch.isalnum() or ch in ("-", "_", "."):
            out.append(ch)
        else:
            out.append("_")
    return ("".join(out).strip("_")) or "dataset"
def _label_from_source(source: Path, raw_dir: Path) -> str:
    src = source
    if not src.is_absolute():
        norm = str(src).replace("\\", "/")
        raw_norm = str(raw_dir).replace("\\", "/").rstrip("/")
        if norm.startswith(raw_norm) or norm.startswith("M3GNet/data/raw/") or norm.startswith("data/raw/"):
            src = Path(norm)
        else:
            src = raw_dir / src
    try:
        rel = src.resolve().relative_to(raw_dir.resolve())
        parts = list(rel.parts)
    except ValueError:
        parts = [src.name]
    if src.is_file():
        parts[-1] = Path(parts[-1]).stem
    return _sanitize("_".join(parts))
if USE_ANI_H5:
    if not H5_FILE.exists():
        raise FileNotFoundError(H5_FILE)
    if len(H5_MOLECULES) < 2:
        raise ValueError("H5_MOLECULES must contain at least two molecule keys.")
    DATASET_SOURCES = list(H5_MOLECULES)
    DATASET_NAMES = [_sanitize(name) for name in DATASET_SOURCES]
else:
    def _resolve_dataset_path(p: str, raw_dir: Path) -> Path:
        path = Path(p)
        if path.is_absolute():
            return path
        norm = str(path).replace("\\", "/")
        raw_norm = str(raw_dir).replace("\\", "/").rstrip("/")
        if norm.startswith(raw_norm) or norm.startswith("M3GNet/data/raw/") or norm.startswith("data/raw/"):
            return path
        return raw_dir / path
    DATASET_PATHS = [_resolve_dataset_path(p, RAW_DIR) for p in DATASET_SOURCES]
    for p in DATASET_PATHS:
        if not p.exists():
            raise FileNotFoundError(p)
    DATASET_NAMES = [_label_from_source(p, RAW_DIR) for p in DATASET_PATHS]
LABELS = DATASET_NAMES
DATASET_A_NAME = DATASET_NAMES[0]
DATASET_B_NAME = DATASET_NAMES[1] if len(DATASET_NAMES) > 1 else None
LABEL_A = LABELS[0]
LABEL_B = LABELS[1] if len(LABELS) > 1 else None
DATASET_A_SOURCE = DATASET_SOURCES[0]
DATASET_B_SOURCE = DATASET_SOURCES[1] if len(DATASET_SOURCES) > 1 else None
# Experiment seeds
SPLIT_SEED = 42
TRAIN_SEED_LIST = [43]
print("Split seed:", SPLIT_SEED)
print("Training seeds:", TRAIN_SEED_LIST)
# Training knobs
PRETRAINED_MODEL = "M3GNet-MatPES-PBE-v2025.1-PES"
CUTOFF = 5.0
INPUT_UNITS = "ev"  # DeepMD datasets are typically in eV/A
SAMPLE_SIZE = 1250
VAL_FRACTION = 0.1
TEST_FRACTION = 0.1
EPOCHS_INDIV = 75
EPOCHS_BY_DATASET = {
    "beta-Li3PS4": 75,
    "gamma-Li3PS4": 75,
}
EPOCHS_COMBINED = 50
BATCH_SIZE = 16
LR = 1e-3
LR_BY_DATASET = {
    "beta-Li3PS4": 1e-3,
    "gamma-Li3PS4": 5e-3,
}
LR_COMBINED = 1e-3
ENERGY_WEIGHT = 1.0
FORCE_WEIGHT = 10
if USE_ANI_H5:
    INPUT_UNITS = "hartree"
CONFIG_PATHS = [f"M3GNet/configs/{name}_quick.yaml" for name in DATASET_NAMES]
RUN_DIRS = [f"M3GNet/runs/{name}" for name in DATASET_NAMES]
CONFIG_A = CONFIG_PATHS[0]
CONFIG_B = CONFIG_PATHS[1] if len(CONFIG_PATHS) > 1 else None
RUN_A = RUN_DIRS[0]
RUN_B = RUN_DIRS[1] if len(RUN_DIRS) > 1 else None
print("Datasets:", ", ".join(DATASET_NAMES))


In [ ]:
# Convert DeepMD folders to .extxyz so prepare_m3gnet_data can read them.
import numpy as np
from ase import Atoms
import ase.io

def _find_deepmd_root(base: Path) -> Path:
    base = Path(base)
    if (base / "type_map.raw").exists() and (base / "type.raw").exists():
        return base
    candidates = []
    for path in base.rglob("type_map.raw"):
        parent = path.parent
        if (parent / "type.raw").exists():
            if list(parent.glob("set.*")) or (parent / "coord.raw").exists():
                candidates.append(parent)
    if not candidates:
        raise FileNotFoundError(f"No DeepMD root found under {base}")
    candidates = sorted(candidates, key=lambda p: str(p))
    if len(candidates) > 1:
        print(f"Multiple DeepMD roots found under {base}, using {candidates[0]}")
    return candidates[0]

def _has_xyz_files(base: Path) -> bool:
    patterns = ("*.xyz", "*.extxyz", "*.xyz.gz", "*.extxyz.gz")
    for pat in patterns:
        if list(base.rglob(pat)):
            return True
    return False

def _read_type_map(path: Path):
    return [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

def _read_type_indices(path: Path):
    return [int(x) for x in path.read_text(encoding="utf-8").split() if x.strip()]

def _load_raw_array(path: Path, width: int):
    arr = np.loadtxt(path)
    if arr.ndim == 1:
        if arr.size % width != 0:
            raise ValueError(f"{path} has {arr.size} values, not divisible by {width}")
        arr = arr.reshape(-1, width)
    return arr

def _load_deepmd_arrays(root: Path, natoms: int):
    set_dirs = sorted([p for p in root.glob("set.*") if p.is_dir()])
    if set_dirs:
        coords, forces, energies, boxes = [], [], [], []
        for d in set_dirs:
            coords.append(np.load(d / "coord.npy"))
            forces.append(np.load(d / "force.npy"))
            energies.append(np.load(d / "energy.npy"))
            boxes.append(np.load(d / "box.npy"))
        coord = np.concatenate(coords, axis=0)
        force = np.concatenate(forces, axis=0)
        energy = np.concatenate(energies, axis=0)
        box = np.concatenate(boxes, axis=0)
        return coord, force, energy, box
    coord = _load_raw_array(root / "coord.raw", natoms * 3)
    force = _load_raw_array(root / "force.raw", natoms * 3)
    energy = np.loadtxt(root / "energy.raw").reshape(-1)
    box = _load_raw_array(root / "box.raw", 9)
    return coord, force, energy, box

def _write_extxyz_from_deepmd(root: Path, output_path: Path):
    type_map = _read_type_map(root / "type_map.raw")
    type_idx = _read_type_indices(root / "type.raw")
    symbols = [type_map[i] for i in type_idx]
    natoms = len(symbols)
    coord, force, energy, box = _load_deepmd_arrays(root, natoms)
    if coord.shape[1] != natoms * 3:
        raise ValueError(f"coord shape {coord.shape} does not match natoms={natoms}")
    atoms_list = []
    for i in range(coord.shape[0]):
        pos = coord[i].reshape(natoms, 3)
        frc = force[i].reshape(natoms, 3)
        cell = box[i].reshape(3, 3)
        atoms = Atoms(symbols, positions=pos, cell=cell, pbc=True)
        atoms.info["energy"] = float(energy[i])
        atoms.arrays["forces"] = frc
        atoms_list.append(atoms)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    ase.io.write(output_path, atoms_list, format="extxyz")
    print(f"Wrote {output_path} with {len(atoms_list)} frames")

if USE_ANI_H5:
    print("Skipping DeepMD conversion (USE_ANI_H5=True).")
else:
    for name in DATASET_SOURCES:
        base = RAW_DIR / name
        if base.is_file():
            print(f"Skipping conversion for {name}: already a file.")
            continue
        if not base.exists():
            raise FileNotFoundError(base)
        if _has_xyz_files(base):
            print(f"Found .xyz/.extxyz under {base}, skipping DeepMD conversion.")
            continue
        extxyz_path = base / f"{name}.extxyz"
        if extxyz_path.exists():
            print(f"Found {extxyz_path}, skipping conversion.")
            continue
        deepmd_root = _find_deepmd_root(base)
        print(f"Converting DeepMD dataset {name} from {deepmd_root}")
        _write_extxyz_from_deepmd(deepmd_root, extxyz_path)


In [ ]:
# Prepare EXTXYZ splits + configs.
import subprocess
import sys
import yaml
from pathlib import Path

cmd = [
    sys.executable,
    "M3GNet/scripts/prepare_m3gnet_data.py",
    "--raw-dir", str(RAW_DIR),
    "--output-dir", "M3GNet/data/prepared",
    "--config-dir", "M3GNet/configs",
    "--sample-size", str(SAMPLE_SIZE),
    "--val-fraction", str(VAL_FRACTION),
    "--test-fraction", str(TEST_FRACTION),
    "--seed", str(SPLIT_SEED),
    "--epochs", str(EPOCHS_INDIV),
    "--model-name", str(PRETRAINED_MODEL),
    "--cutoff", str(CUTOFF),
    "--batch-size", str(BATCH_SIZE),
    "--lr", str(LR),
    "--energy-weight", str(ENERGY_WEIGHT),
    "--force-weight", str(FORCE_WEIGHT),
    "--input-units", str(INPUT_UNITS),
    "--include-combined",
]
for ds in DATASET_SOURCES:
    cmd += ["--dataset", str(ds)]
if USE_ANI_H5:
    cmd += ["--h5-file", str(H5_FILE)]

subprocess.check_call(cmd)

combined_cfg_path = Path("M3GNet/configs/combined_quick.yaml")
if combined_cfg_path.exists():
    cfg = yaml.safe_load(combined_cfg_path.read_text(encoding="utf-8"))
    cfg.setdefault("train", {})["epochs"] = EPOCHS_COMBINED
    combined_cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    print("Updated epochs in", combined_cfg_path)


In [ ]:
# Write a combined-val eval config that matches the selected model.
import yaml
from pathlib import Path

cfg = {
    "model": {"pretrained_name": PRETRAINED_MODEL, "cutoff": CUTOFF},
    "train": {
        "seed": SPLIT_SEED,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "energy_weight": ENERGY_WEIGHT,
        "force_weight": FORCE_WEIGHT,
        "stress_weight": 0.0,
        "decay_steps": 1000,
        "decay_alpha": 0.01,
        "num_workers": 0,
    },
    "data": {
        "val_path": "data/prepared/combined_val.extxyz",
        "test_path": "data/prepared/combined_test.extxyz",
        "cache_dir": "data/cache/combined_val",
    },
    "output": {"run_dir": "runs/combined_val_eval"},
}
out_path = Path("M3GNet/configs/combined_val_eval.yaml")
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print("Wrote", out_path)


In [ ]:
# Helpers for running commands and capturing times/metrics
import re
import subprocess
from pathlib import Path
import yaml

TRAINING_TIMES = {}
MERGE_TIMES = {}

def run_cmd_stream(cmd):
    print(" ".join(str(part) for part in cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    output = []
    for line in proc.stdout:
        print(line, end="")
        output.append(line)
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"Command failed with exit code {ret}")
    return "".join(output)

def update_train_seed(cfg_path, seed, epochs=None, lr=None):
    cfg = yaml.safe_load(Path(cfg_path).read_text(encoding="utf-8"))
    train_cfg = cfg.setdefault("train", {})
    train_cfg["seed"] = int(seed)
    if epochs is not None:
        train_cfg["epochs"] = int(epochs)
    if lr is not None:
        train_cfg["lr"] = float(lr)
    Path(cfg_path).write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")

def run_training(config, workdir, epochs, seed, lr_override=None):
    update_train_seed(config, seed, epochs=epochs, lr=lr_override)
    cmd = [
        "python",
        "M3GNet/scripts/train_m3gnet.py",
        "--config", str(config),
        "--workdir", str(workdir),
        "--device", "auto",
        "--plot",
        "--epochs", str(epochs),
    ]
    if lr_override is not None:
        cmd += ["--lr", str(lr_override)]
    output = run_cmd_stream(cmd)
    match = re.search(r"Training time: ([0-9.]+)s", output)
    if not match:
        raise ValueError("Training time not found in output.")
    seconds = float(match.group(1))
    TRAINING_TIMES[str(workdir)] = seconds
    Path(workdir).mkdir(parents=True, exist_ok=True)
    Path(workdir, "training_time.txt").write_text(str(seconds), encoding="utf-8")
    return seconds

def parse_kv_metrics(path):
    metrics = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or "=" not in line:
            continue
        key, val = line.split("=", 1)
        key = key.strip()
        val = val.strip()
        try:
            metrics[key] = float(val)
        except ValueError:
            continue
    return metrics

def eval_checkpoint_metrics(checkpoint, config, split, output_csv):
    cmd = [
        "python",
        "M3GNet/scripts/evaluate_m3gnet.py",
        "--split", split,
        "--checkpoint", str(checkpoint),
        "--config", str(config),
        "--output-csv", str(output_csv),
    ]
    run_cmd_stream(cmd)
    import csv
    with open(output_csv, newline="", encoding="utf-8") as handle:
        row = next(csv.DictReader(handle))
    metrics = {}
    for key, value in row.items():
        if key in ("checkpoint", "config"):
            continue
        if value in (None, ""):
            continue
        try:
            val = float(value)
        except ValueError:
            continue
        if split == "test" and key.startswith("val_"):
            metrics[f"test_{key[4:]}"] = val
        else:
            metrics[key] = val
    return metrics

def get_training_time(run_dir):
    if str(run_dir) in TRAINING_TIMES:
        return TRAINING_TIMES[str(run_dir)]
    path = Path(run_dir) / "training_time.txt"
    if path.exists():
        try:
            return float(path.read_text(encoding="utf-8").strip())
        except ValueError:
            return None
    return None


In [ ]:
# Train per-dataset models (fixed split, varying training seed)
for seed in TRAIN_SEED_LIST:
    print(f"=== Training seed {seed} ===")
    run_dirs = [f"{base}_seed{seed}" for base in RUN_DIRS]
    for name, cfg, run_dir in zip(DATASET_NAMES, CONFIG_PATHS, run_dirs):
        lr = LR_BY_DATASET.get(name, LR) if LR_BY_DATASET else LR
        epochs = EPOCHS_BY_DATASET.get(name, EPOCHS_INDIV) if EPOCHS_BY_DATASET else EPOCHS_INDIV
        train_time = run_training(cfg, run_dir, epochs, seed, lr)
        print(f"Training time {name} (s):", train_time)


In [ ]:
# (Handled above)


In [ ]:
# Train combined (fixed split, varying training seed)
for seed in TRAIN_SEED_LIST:
    run_dir = f"M3GNet/runs/combined_seed{seed}"
    train_time_combined = run_training("M3GNet/configs/combined_quick.yaml", run_dir, EPOCHS_COMBINED, seed, LR_COMBINED)
    print(f"Training time combined seed {seed} (s):", train_time_combined)


In [ ]:
# Resolve checkpoints from training runs (best_chk / last / newest).
from pathlib import Path

def pick_checkpoint(run_dir):
    run_dir = Path(run_dir)
    best = run_dir / "best_chk.ckpt"
    if best.exists():
        return str(best)
    ckpt_dir = run_dir / "checkpoints"
    last = ckpt_dir / "last.ckpt"
    if last.exists():
        return str(last)
    ckpts = sorted(ckpt_dir.glob("*.ckpt"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not ckpts:
        raise FileNotFoundError(f"No checkpoints found under {ckpt_dir}")
    return str(ckpts[0])

# Resolve per-seed checkpoints.
ENABLE_SWITCH = len(DATASET_NAMES) >= 2
SEED_INFO = {}
for seed in TRAIN_SEED_LIST:
    run_dirs = [f"{base}_seed{seed}" for base in RUN_DIRS]
    ckpt_list = [pick_checkpoint(run) for run in run_dirs]
    source_checkpoints = [f"{label}={ckpt}" for label, ckpt in zip(LABELS, ckpt_list)]
    seed_info = {
        "run_dirs": run_dirs,
        "ckpt_list": ckpt_list,
        "source_checkpoints": source_checkpoints,
        "merge_checkpoints": list(ckpt_list),
        "merge_configs": [str(Path(run) / "config.yaml") for run in run_dirs],
        "combined_run": f"M3GNet/runs/combined_seed{seed}",
        "mean_ckpt": f"M3GNet/runs/merged/mean_seed{seed}.ckpt",
        "closed_form_ckpt": f"M3GNet/runs/merged/closed_form_individual_seed{seed}.ckpt",
    }
    SEED_INFO[seed] = seed_info
    print(f"Using checkpoints for seed {seed}:")
    for run, ckpt in zip(run_dirs, ckpt_list):
        print(f"  {run}: {ckpt}")


In [ ]:
# Evaluate trained models on their test sets (per training seed)
import subprocess
from pathlib import Path

for seed in TRAIN_SEED_LIST:
    run_dirs = [Path(run) for run in SEED_INFO[seed]["run_dirs"]] + [Path(SEED_INFO[seed]["combined_run"])]
    cmd = ["python", "M3GNet/scripts/evaluate_m3gnet.py", "--split", "test"]
    for run in run_dirs:
        cmd += ["--run-dir", str(run)]
    print(" ".join(cmd))
    subprocess.check_call(cmd)


In [ ]:
# Mean-merge checkpoints and evaluate on combined test (per training seed)
import subprocess

for seed in TRAIN_SEED_LIST:
    merge_ckpts = SEED_INFO[seed]["merge_checkpoints"]
    if len(merge_ckpts) < 2:
        raise ValueError("Need at least two checkpoints to merge.")
    mean_ckpt = SEED_INFO[seed]["mean_ckpt"]
    cmd = ["python", "M3GNet/scripts/merge_m3gnet_checkpoints.py"]
    for ckpt in merge_ckpts:
        cmd += ["--ckpt", str(ckpt)]
    cmd += ["--output-ckpt", str(mean_ckpt)]
    print(" ".join(cmd))
    subprocess.check_call(cmd)

    cmd = ["python", "M3GNet/scripts/evaluate_m3gnet.py", "--split", "test", "--checkpoint", str(mean_ckpt), "--config", "M3GNet/configs/combined_val_eval.yaml"]
    print(" ".join(cmd))
    subprocess.check_call(cmd)


In [ ]:
# Closed-form individual merge (prints merge time) and evaluation (per training seed)
import re

for seed in TRAIN_SEED_LIST:
    merge_ckpts = SEED_INFO[seed]["merge_checkpoints"]
    merge_cfgs = SEED_INFO[seed]["merge_configs"]
    if len(merge_ckpts) != len(merge_cfgs):
        raise ValueError("MERGE_CHECKPOINTS and MERGE_CONFIGS must be the same length.")
    out_ckpt = SEED_INFO[seed]["closed_form_ckpt"]
    cmd = ["python", "M3GNet/scripts/merge_closed_form_individual_m3gnet.py"]
    for ckpt in merge_ckpts:
        cmd += ["--checkpoint", str(ckpt)]
    for cfg in merge_cfgs:
        cmd += ["--config", str(cfg)]
    cmd += ["--batch-size", "64", "--output-ckpt", str(out_ckpt)]
    merge_output = run_cmd_stream(cmd)

    match = re.search(r"Closed-form individual merge compute time: ([0-9.]+)s", merge_output)
    if match:
        MERGE_TIMES[seed] = float(match.group(1))
        print(f"Closed-form merge time seed {seed} (s):", MERGE_TIMES[seed])
    else:
        print(f"Warning: merge time not found in output for seed {seed}.")

    cmd = ["python", "M3GNet/scripts/evaluate_m3gnet.py", "--split", "test", "--checkpoint", str(out_ckpt), "--config", "M3GNet/configs/combined_val_eval.yaml"]
    run_cmd_stream(cmd)


In [ ]:
if not ENABLE_SWITCH:
    print("Skipping switch-embedding step (requires at least 2 datasets).")
else:
    # Switched-embedding evaluation for the closed-form merged model (test)
    from subprocess import check_call
    import sys

    for seed in TRAIN_SEED_LIST:
        source_args = []
        for entry in SEED_INFO[seed]["source_checkpoints"]:
            source_args += ["--source-checkpoint", str(entry)]
        check_call([
            sys.executable,
            "M3GNet/scripts/evaluate_switch_embeddings_m3gnet.py",
            "--split", "test",
            "--config", "M3GNet/configs/combined_val_eval.yaml",
            "--checkpoint", str(SEED_INFO[seed]["closed_form_ckpt"]),
        ] + source_args)


In [ ]:
# Collect experiment metrics and write summary CSVs per training seed
import csv
from pathlib import Path
all_summary_rows = []
summary_paths = []
for seed in TRAIN_SEED_LIST:
    summary_rows = []
    run_dirs = SEED_INFO[seed]["run_dirs"]
    checkpoint_defs = [
        (name, Path(run) / "best_chk.ckpt", Path(run) / "config.yaml", run)
        for name, run in zip(DATASET_NAMES, run_dirs)
    ]
    checkpoint_defs.append((
        "combined",
        Path(SEED_INFO[seed]["combined_run"]) / "best_chk.ckpt",
        Path(SEED_INFO[seed]["combined_run"]) / "config.yaml",
        SEED_INFO[seed]["combined_run"],
    ))
    for name, ckpt_path, cfg_path, run_dir in checkpoint_defs:
        metrics_csv = Path(f"M3GNet/runs/metrics_{name}_test_seed{seed}.csv")
        metrics = eval_checkpoint_metrics(ckpt_path, cfg_path, "test", metrics_csv)
        metrics = metrics
        summary_rows.append({
            "seed": seed,
            "model_label": name,
            "checkpoint": str(ckpt_path),
            "eval_type": "standard",
            "training_time_seconds": get_training_time(run_dir),
            "merge_time_seconds": None,
            "test_Energy_MAE": metrics.get("test_Energy_MAE"),
            "test_Energy_RMSE": metrics.get("test_Energy_RMSE"),
            "test_Force_MAE": metrics.get("test_Force_MAE"),
            "test_Force_RMSE": metrics.get("test_Force_RMSE"),
        })
    mean_ckpt = Path(SEED_INFO[seed]["mean_ckpt"])
    mean_metrics_csv = Path(f"M3GNet/runs/metrics_mean_merge_test_seed{seed}.csv")
    mean_metrics = eval_checkpoint_metrics(mean_ckpt, "M3GNet/configs/combined_val_eval.yaml", "test", mean_metrics_csv)
    mean_metrics = mean_metrics
    summary_rows.append({
        "seed": seed,
        "model_label": "mean_merge",
        "checkpoint": str(mean_ckpt),
        "eval_type": "mean_merge",
        "training_time_seconds": None,
        "merge_time_seconds": None,
        "test_Energy_MAE": mean_metrics.get("test_Energy_MAE"),
        "test_Energy_RMSE": mean_metrics.get("test_Energy_RMSE"),
        "test_Force_MAE": mean_metrics.get("test_Force_MAE"),
        "test_Force_RMSE": mean_metrics.get("test_Force_RMSE"),
    })
    merged_ckpt = Path(SEED_INFO[seed]["closed_form_ckpt"])
    merged_metrics_csv = Path(f"M3GNet/runs/metrics_closed_form_individual_test_seed{seed}.csv")
    merged_metrics = eval_checkpoint_metrics(merged_ckpt, "M3GNet/configs/combined_val_eval.yaml", "test", merged_metrics_csv)
    merged_metrics = merged_metrics
    summary_rows.append({
        "seed": seed,
        "model_label": "closed_form_individual",
        "checkpoint": str(merged_ckpt),
        "eval_type": "standard",
        "training_time_seconds": None,
        "merge_time_seconds": MERGE_TIMES.get(seed),
        "test_Energy_MAE": merged_metrics.get("test_Energy_MAE"),
        "test_Energy_RMSE": merged_metrics.get("test_Energy_RMSE"),
        "test_Force_MAE": merged_metrics.get("test_Force_MAE"),
        "test_Force_RMSE": merged_metrics.get("test_Force_RMSE"),
    })
    if ENABLE_SWITCH:
        switch_metrics_path = Path(f"M3GNet/runs/closed_form_individual_switch_test_seed{seed}.txt")
        if not switch_metrics_path.exists():
            source_args = []
            for entry in SEED_INFO[seed]["source_checkpoints"]:
                source_args += ["--source-checkpoint", str(entry)]
            run_cmd_stream([
                "python",
                "M3GNet/scripts/evaluate_switch_embeddings_m3gnet.py",
                "--split", "test",
                "--config", "M3GNet/configs/combined_val_eval.yaml",
                "--checkpoint", str(merged_ckpt),
                "--save", str(switch_metrics_path),
            ] + source_args)
        switch_metrics = parse_kv_metrics(switch_metrics_path)
        summary_rows.append({
            "seed": seed,
            "model_label": "closed_form_individual",
            "checkpoint": str(merged_ckpt),
            "eval_type": "switch",
            "training_time_seconds": None,
            "merge_time_seconds": None,
            "test_Energy_MAE": switch_metrics.get("test_Energy_MAE"),
            "test_Energy_RMSE": switch_metrics.get("test_Energy_RMSE"),
            "test_Force_MAE": switch_metrics.get("test_Force_MAE"),
            "test_Force_RMSE": switch_metrics.get("test_Force_RMSE"),
        })
    else:
        print("Skipping switch-embedding evaluation (requires at least 2 datasets).")
    summary_path = Path(f"M3GNet/runs/experiment_summary_seed{seed}.csv")
    fieldnames = ["seed", "model_label", "checkpoint", "eval_type", "training_time_seconds", "merge_time_seconds", "test_Energy_MAE", "test_Energy_RMSE", "test_Force_MAE", "test_Force_RMSE"]
    with summary_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in summary_rows:
            writer.writerow(row)
    print(f"Wrote summary CSV to {summary_path}")
    summary_paths.append(summary_path)
    all_summary_rows.extend(summary_rows)


In [ ]:
if not ENABLE_SWITCH:
    print("Skipping switch-embedding step (requires at least 2 datasets).")
else:
    # Fine-tune final readout + last N blocks (per (epochs, limit) val selection)
    from pathlib import Path
    import subprocess
    import time
    import csv
    LAST_N_BLOCKS = 3
    EPOCH_LIMIT_PAIRS = [
        (10, 125),
        (10, 250),
        (10, 500),
        (10, 1000),
        (20, 125),
        (20, 250),
        (20, 500),
        (20, 750),
        (30, 125),
        (30, 250),
        (30, 500),
        (40, 250),
    ]
    LR_LIST = [5e-5, 1e-4, 5e-4, 1e-3, 5e-3]
    def run_cmd(cmd):
        print(" ".join(str(part) for part in cmd))
        subprocess.check_call(cmd)
    def parse_metric(path, key):
        text = Path(path).read_text(encoding="utf-8")
        for line in text.splitlines():
            line = line.strip()
            if line.startswith(key + "="):
                return float(line.split("=", 1)[1])
        raise ValueError(f"{key} not found in {path}")
    for seed in TRAIN_SEED_LIST:
        RESULTS_DIR = Path(f"M3GNet/runs/finetune_last_block_grid_seed{seed}")
        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        source_args = []
        for entry in SEED_INFO[seed]["source_checkpoints"]:
            source_args += ["--source-checkpoint", str(entry)]
        summary_rows = []
        for epochs, limit in EPOCH_LIMIT_PAIRS:
            best_loss = (float("inf"), None, None)
            best_duration = None
            for lr in LR_LIST:
                out_ckpt = RESULTS_DIR / f"closed_form_individual_ft_last_block_ep{epochs}_lim{limit}_lr{lr}.ckpt"
                metrics_path = RESULTS_DIR / f"val_metrics_ep{epochs}_lim{limit}_lr{lr}.txt"
                print()
                print(f"=== Fine-tune (last {LAST_N_BLOCKS} blocks): seed={seed} epochs={epochs} limit={limit} lr={lr} ===")
                start = time.perf_counter()
                run_cmd([
                    "python",
                    "M3GNet/scripts/switch_finetune_energy_readout_last_block_m3gnet.py",
                    "--config", f"{SEED_INFO[seed]['combined_run']}/config.yaml",
                    "--checkpoint", str(SEED_INFO[seed]["closed_form_ckpt"]),
                    "--epochs", str(epochs),
                    "--limit", str(limit),
                    "--lr", str(lr),
                    "--force-weight", str(FORCE_WEIGHT),
                    "--energy-weight", str(ENERGY_WEIGHT),
                    "--seed", str(seed),
                    "--last-n-blocks", str(LAST_N_BLOCKS),
                    "--output", str(out_ckpt),
                ] + source_args)
                duration = time.perf_counter() - start
                print(f"=== Validate: {out_ckpt} ===")
                run_cmd([
                    "python",
                    "M3GNet/scripts/evaluate_switch_embeddings_m3gnet.py",
                    "--split", "val",
                    "--config", "M3GNet/configs/combined_val_eval.yaml",
                    "--checkpoint", str(out_ckpt),
                    "--save", str(metrics_path),
                ] + source_args)
                val_loss = parse_metric(metrics_path, "val_Total_Loss")
                if val_loss < best_loss[0]:
                    best_loss = (val_loss, lr, out_ckpt)
                    best_duration = duration
            print(f"Best val loss for seed={seed} epochs={epochs} limit={limit}: lr={best_loss[1]} val_Total_Loss={best_loss[0]:.6f}")
            test_energy_mae = None
            test_energy_rmse = None
            test_force_mae = None
            test_force_rmse = None
            if best_loss[2] is not None:
                print()
                print(f"=== Test eval (best_val_loss) seed={seed} epochs={epochs} limit={limit} lr={best_loss[1]} ===")
                test_metrics = RESULTS_DIR / f"test_best_val_loss_ep{epochs}_lim{limit}_lr{best_loss[1]}.txt"
                run_cmd([
                    "python",
                    "M3GNet/scripts/evaluate_switch_embeddings_m3gnet.py",
                    "--split", "test",
                    "--config", "M3GNet/configs/combined_val_eval.yaml",
                    "--checkpoint", str(best_loss[2]),
                    "--save", str(test_metrics),
                ] + source_args)
                test_energy_mae = parse_metric(test_metrics, "test_Energy_MAE")
                test_energy_rmse = parse_metric(test_metrics, "test_Energy_RMSE")
                test_force_mae = parse_metric(test_metrics, "test_Force_MAE")
                test_force_rmse = parse_metric(test_metrics, "test_Force_RMSE")
            summary_rows.append({
                "epochs": epochs,
                "limit": limit,
                "best_lr": best_loss[1],
                "finetune_seconds": None if best_duration is None else round(best_duration, 2),
                "test_energy_mae": test_energy_mae,
                "test_energy_rmse": test_energy_rmse,
                "test_force_mae": test_force_mae,
                "test_force_rmse": test_force_rmse,
            })
        summary_path = RESULTS_DIR / f"grid_summary_seed{seed}.csv"
        with summary_path.open("w", newline="", encoding="utf-8") as handle:
            fieldnames = ["epochs", "limit", "best_lr", "finetune_seconds", "test_energy_mae", "test_energy_rmse", "test_force_mae", "test_force_rmse"]
            writer = csv.DictWriter(handle, fieldnames=fieldnames)
            writer.writeheader()
            for row in summary_rows:
                writer.writerow(row)
        print()
        print(f"Wrote summary CSV to {summary_path}")
        print()
        print("Summary table:")
        header = ["epochs", "limit", "best_lr", "finetune_seconds", "test_energy_mae", "test_energy_rmse", "test_force_mae", "test_force_rmse"]
        print("	".join(header))
        for row in summary_rows:
            print("	".join(str(row.get(key, "")) for key in header))


In [ ]:
# Aggregate mean/std across training seeds (metrics are in eV)
import pandas as pd
from pathlib import Path
summary_frames = []
for seed in TRAIN_SEED_LIST:
    path_csv = Path(f"M3GNet/runs/experiment_summary_seed{seed}.csv")
    if path_csv.exists():
        summary_frames.append(pd.read_csv(path_csv))
if summary_frames:
    df = pd.concat(summary_frames, ignore_index=True)
    numeric_cols = [
        col for col in df.columns
        if col not in ("seed", "model_label", "checkpoint", "eval_type")
    ]
    grouped = df.groupby(["model_label", "eval_type"], dropna=False)[numeric_cols]
    mean_df = grouped.mean().add_suffix("_mean")
    std_df = grouped.std(ddof=0).add_suffix("_std")
    out = pd.concat([mean_df, std_df], axis=1).reset_index()
    out_path = Path("M3GNet/runs/experiment_summary_mean_std.csv")
    out.to_csv(out_path, index=False)
    print("Wrote", out_path)
else:
    print("No experiment_summary_seed*.csv found to aggregate.")
grid_frames = []
for seed in TRAIN_SEED_LIST:
    path_csv = Path(f"M3GNet/runs/finetune_last_block_grid_seed{seed}/grid_summary_seed{seed}.csv")
    if path_csv.exists():
        grid_frames.append(pd.read_csv(path_csv))
if grid_frames:
    grid_df = pd.concat(grid_frames, ignore_index=True)
    metric_cols = ["finetune_seconds", "test_energy_mae", "test_energy_rmse", "test_force_mae", "test_force_rmse"]
    grouped = grid_df.groupby(["epochs", "limit"], dropna=False)
    mean_df = grouped[metric_cols].mean().add_suffix("_mean")
    std_df = grouped[metric_cols].std(ddof=0).add_suffix("_std")
    best_lr_mode = grouped["best_lr"].agg(lambda s: s.mode().iloc[0] if not s.mode().empty else None)
    out = pd.concat([mean_df, std_df, best_lr_mode.rename("best_lr_mode")], axis=1).reset_index()
    out_path = Path("M3GNet/runs/grid_summary_mean_std.csv")
    out.to_csv(out_path, index=False)
    print("Wrote", out_path)
else:
    print("No grid_summary_seed*.csv found to aggregate.")
